In [1]:
import pandas as pd
from metadata_schemas.ai_taxonomy import get_ieee_tax, get_ieeefull_tax
from pprint import pprint
IEEE_TAX = get_ieee_tax()
IEEEFULL_TAX = get_ieeefull_tax()

In [2]:
ieee_strings = str(IEEE_TAX).split("'")[1::2]
ieeefull_strings = str(IEEEFULL_TAX).split("'")[1::2]

In [3]:
n = 1
RESULTS_DIR = "all_results"


from pathlib import Path
import json
import re

RESULTS_DIR = Path(RESULTS_DIR)

ieee_paths: dict[str, list[str]] = {}
ieeefull_paths: dict[str, list[str]] = {}

for json_path in RESULTS_DIR.glob('*.json'):
    key = json_path.stem

    if key.startswith("ieee_") or key.startswith("fullieee_"):



        with json_path.open() as handle:
            payload = json.load(handle)

        path_value = None
        for value in payload.values():
            if isinstance(value, dict):
                try:
                    path_value = [*value["category"]]
                    path_value = [string.replace("_"," ") for string in path_value] # slight syntax adjustment
                    #path_value = [*value["appl"]]
                except KeyError as e:
                    print(value)
                    raise e
                #path_value = value

        if path_value is None:
            continue

        if key.startswith("fullieee"):
            ieeefull_paths[key] = path_value
        elif key.startswith("ieee_"):
            ieee_paths[key] = path_value



In [4]:
len(ieee_paths), len(ieeefull_paths)

(1405, 3405)

In [5]:
df = pd.read_csv("/mnt/data/upcast/data/trend_analysis/ieee_ai_cl10.csv", index_col=0)
print(len(df))
df = df.drop_duplicates()
print(len(df))

full_df = pd.read_csv("/mnt/data/upcast/data/trend_analysis/ieee_full_cl10.csv", index_col=0)
print(len(full_df))
full_df = full_df.drop_duplicates()
print(len(full_df))



1552
1510
7681
7359


In [6]:
#for key in ieee_paths:
#    path = ieee_paths[key]
#    if 'Learning (artificial intelligence)' in path:
#        print(key, path)

In [7]:
# make term-to-path dict

def make_term_to_path_dict(term_to_path_dict, subtree, current_path):
    if subtree is None:
        return
    for term in subtree:
        term_path = [*current_path, term]
        #print(term, term_path, subtree[term])
        if term in term_to_path_dict:
            term_to_path_dict[term].append(term_path)
        else:
            term_to_path_dict[term] = [term_path]
        make_term_to_path_dict(term_to_path_dict, subtree[term], term_path)


term_to_path_dict = {}
make_term_to_path_dict(term_to_path_dict, IEEE_TAX, [])

full_term_to_path_dict = {}
make_term_to_path_dict(full_term_to_path_dict, IEEEFULL_TAX, [])

In [8]:
ignore_first=True#False
def count_overlap(pred, gt):
    o = 0
    for i in range(min(len(gt), len(pred))):
        if pred[i]==gt[i]:
            o+=1
        else:
            break
    return o
def calc_recall(pred, gt):
    if ignore_first:
        pred, gt = pred[1:], gt[1:]
    if len(gt) == 0:
        return 1
    return count_overlap(pred, gt) / len(gt)
def calc_precision(pred, gt):
    if ignore_first:
        pred, gt = pred[1:], gt[1:]
    if pred and pred[-1] == "Other":
        pred = pred[:-1]
    o = count_overlap(pred, gt)
    if len(pred) > o:
        return o/(o+1)
    else:
        return 1
def calc_f1(r,p):
    if r==0 and p==0:
        return 0
    return 2*p*r/(r+p)

In [9]:
df.loc["ieee_5047", "IEEE Terms"]


'Fuzzy neural networks;Cellular neural networks;Neurofeedback;State feedback;Feedforward systems;Cellular networks;Neural networks;Fuzzy logic;Delay effects;Computer networks'

In [10]:
results = pd.DataFrame(columns=["recall", "precision", "f1"])
for key in ieee_paths:
    pred = ieee_paths[key]
    #gts = []
    gt_terms = df.loc[key, "IEEE Terms"]
    #print(gt_terms)
    gt_terms = gt_terms.split(";")

    #print(pred)
    #print(gt_terms)
    recalls = []
    precisions = []
    for gt_term in gt_terms:
        if gt_term in ieee_strings:
            gt_paths = term_to_path_dict[gt_term]
            for gt_path in gt_paths:
                recalls.append(calc_recall(pred, gt_path))
                precisions.append(calc_precision(pred, gt_path))
                #print(recalls[-1], precisions[-1], pred, gt_path)
    if len(recalls) == 0:
        continue # 
    recall = max(recalls)
    precision = max(precisions)
    results.loc[key] = (recall, precision, calc_f1(recall,precision))


In [11]:
results

,recall,precision,f1
ieee_40938,0.666667,1.000000,0.8
ieee_16802,1.000000,1.000000,1.0
ieee_4042,0.500000,0.500000,0.5
ieee_43881,0.333333,0.500000,0.4
ieee_1785,1.000000,1.000000,1.0
...,...,...,...
ieee_27010,0.000000,0.000000,0.0
ieee_3580,0.000000,0.000000,0.0
ieee_15688,1.000000,1.000000,1.0
ieee_52407,0.500000,0.500000,0.5


In [12]:
results.mean()

recall       0.477817
precision    0.506050
f1           0.450629
dtype: float64

# Full tax

In [13]:
full_results = pd.DataFrame(columns=["recall", "precision", "f1"])
for key in ieeefull_paths:
    pred = ieeefull_paths[key]
    #gts = []
    #print(key)
    gt_terms = full_df.loc[key, "IEEE Terms"]
    #print(gt_terms)
    gt_terms = gt_terms.split(";")

    #print(pred)
    #print(gt_terms)
    recalls = []
    precisions = []
    for gt_term in gt_terms:
        if gt_term in ieeefull_strings:
            gt_paths = full_term_to_path_dict[gt_term]
            for gt_path in gt_paths:
                recalls.append(calc_recall(pred, gt_path))
                precisions.append(calc_precision(pred, gt_path))
                #print(recalls[-1], precisions[-1], pred, gt_path)
    if len(recalls) == 0:
        continue # 
    recall = max(recalls)
    precision = max(precisions)
    full_results.loc[key] = (recall, precision, calc_f1(recall,precision))


In [14]:
full_results

,recall,precision,f1
fullieee_36325,0.0,1.0,0.0
fullieee_22732,0.5,0.5,0.5
fullieee_20107,0.0,0.0,0.0
fullieee_16287,0.0,1.0,0.0
fullieee_31647,0.0,0.0,0.0
...,...,...,...
fullieee_43178,0.0,1.0,0.0
fullieee_43391,0.0,1.0,0.0
fullieee_12919,0.0,1.0,0.0
fullieee_32103,0.0,1.0,0.0


In [15]:
full_results.mean()

recall       0.156920
precision    0.444167
f1           0.143344
dtype: float64

# Compare to other methods

## Similarity

In [12]:
embedding_model_id = "all-MiniLM-L6-v2"
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
embedding_model = SentenceTransformer(embedding_model_id, device="cuda:1" if torch.cuda.device_count()>1 else "cuda:0")


/home/sondre/miniconda3/envs/upcast/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [13]:
ieee_string_embeddings = embedding_model.encode(ieee_strings)
paper_texts = ("Title: "+df["Document Title"]+ "\nAbstract: "+df["Abstract"]).to_list()
paper_embeddings = embedding_model.encode(paper_texts)


In [14]:
similarities = util.cos_sim(paper_embeddings, ieee_string_embeddings)

In [15]:
normalise = False
if normalise:
    sim_score = (similarities - similarities.mean(axis=0)) / similarities.std(axis=0)
else:
    sim_score = similarities

similarity_preds = [ieee_strings[i] for i in torch.argmax(sim_score, dim=1)]

In [26]:
p_lens = {0:0, 1:0, 2:0, 3:0, 4:0, 5:0}
sim_results = pd.DataFrame(columns=["recall", "precision", "f1"])
for i in range(len(similarity_preds)):
    pred = similarity_preds[i]
    preds = term_to_path_dict[pred]
    p_lens[len(preds)] += 1

    gt_terms = df.iloc[i]["IEEE Terms"]
    #print(gt_terms)
    gt_terms = gt_terms.split(";")

    #print(pred)
    #print(gt_terms)
    recalls = []
    precisions = []
    for pred in preds:
        for gt_term in gt_terms:
            if gt_term in ieee_strings:
                gt_paths = term_to_path_dict[gt_term]
                for gt_path in gt_paths:
                    recalls.append(calc_recall(pred, gt_path))
                    precisions.append(calc_precision(pred, gt_path))
                    #print(recalls[-1], precisions[-1], pred, gt_path)
    if len(recalls) == 0:
        continue # 
    recall = max(recalls)
    precision = max(precisions)
    sim_results.loc[i] = (recall, precision, calc_f1(recall,precision))
print(p_lens)

{0: 0, 1: 1496, 2: 14, 3: 0, 4: 0, 5: 0}


In [27]:
#similarity_preds

In [17]:
results.mean()

recall       0.477817
precision    0.506050
f1           0.450629
dtype: float64

In [39]:
sim_results.mean()

recall       0.799338
precision    0.785430
f1           0.787241
dtype: float64

In [40]:
sim_results

,recall,precision,f1
0,1.000000,1.000000,1.000000
1,1.000000,1.000000,1.000000
2,1.000000,1.000000,1.000000
3,1.000000,1.000000,1.000000
4,1.000000,1.000000,1.000000
...,...,...,...
1505,0.000000,0.000000,0.000000
1506,0.333333,0.500000,0.400000
1507,0.500000,0.500000,0.500000
1508,0.333333,0.500000,0.400000


In [41]:
df

,Abstract,IEEE Term,Document Title,PDF Link,Author Keywords,IEEE Terms,Mesh_Terms
index,,,,,,,
ieee_87,"In recent years, deep learning-based medical i...",Active learning,Dual-Criteria Active Learning for Medical Imag...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,active learning;medical image segmentation;dee...,Training;Measurement;Deep learning;Image segme...,NaN
ieee_88,Federated Learning (FL) is a learning paradigm...,Active learning,LG-FAL: Locality-customized GSA Federated Acti...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,Federated active learning;active learning;para...,Adaptation models;Accuracy;Uncertainty;Federat...,NaN
ieee_89,This paper introduces a novel two-stage active...,Active learning,Combining X-Vectors and Bayesian Batch Active ...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,Active learning;batch active learning;Bayesian...,Data models;Training;Active learning;Uncertain...,NaN
ieee_90,Artificial Intelligence in medicine can improv...,Active learning,Efficient Selection of Rare Pathology Samples ...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,Artificial Intelligence;Active Learning;Medica...,Pathology;Computer vision;Circuits and systems...,NaN
ieee_91,Predicting blood-brain barrier (BBB) permeabil...,Active learning,Active Learning for Predicting Drug Permeabili...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,Active Learning;Dynamic Sampling;Scaffold Spli...,Drugs;Training;Accuracy;Uncertainty;Active lea...,NaN
...,...,...,...,...,...,...,...
ieee_54328,We propose a saliency-guided Transformer-based...,Weak supervision,Saliency-Guided and Feature Fusion with Transf...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,image semantic segmentation;saliency guidance;...,Location awareness;Head;Weak supervision;Autom...,NaN
ieee_54329,Weakly supervised semantic segmentation aims t...,Weak supervision,Robust Weakly Supervised Semantic Segmentation...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,Weakly supervised learning;semantic segmentati...,Training;Technological innovation;Weak supervi...,NaN
ieee_54330,"Recently, there appears aspect-based review sp...",Weak supervision,Detecting aspect-based review spam based on fu...,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...,Aspect-based Review Spam;Fuzzy Ellipsoid Numbe...,Sentiment analysis;Weak supervision;Machine le...,NaN


In [19]:
similarities - similarities.mean(axis=0)

tensor([[-0.0800, -0.0605, -0.0777,  ..., -0.0132, -0.0542, -0.0212],
        [ 0.0098,  0.0176,  0.0101,  ...,  0.0550,  0.0244, -0.0278],
        [-0.0627, -0.0273, -0.0836,  ...,  0.0404, -0.0453, -0.0602],
        ...,
        [ 0.0012, -0.0225,  0.1403,  ..., -0.0079, -0.0189, -0.0240],
        [-0.1030, -0.0875, -0.0991,  ...,  0.0797, -0.0559, -0.0519],
        [-0.0973, -0.0542,  0.0104,  ...,  0.0190, -0.0415,  0.0119]])

In [20]:
similarities.mean(axis=0)
#similarities.std(axis=0)

tensor([ 0.1599,  0.1941,  0.1174,  0.1672,  0.1901,  0.0977,  0.1200,  0.1123,
         0.0679,  0.1204,  0.1145,  0.1540,  0.1302,  0.1625,  0.0667,  0.1787,
         0.1788,  0.0497,  0.1957,  0.0801,  0.1563,  0.1207,  0.1224,  0.1384,
         0.1700,  0.1401,  0.1255,  0.1759,  0.1010,  0.1139,  0.1003,  0.1200,
         0.1442,  0.1703,  0.1361,  0.1074,  0.1942,  0.1931,  0.1596,  0.1123,
         0.1578,  0.1591,  0.1825,  0.1205,  0.1493,  0.1324,  0.1728,  0.1575,
        -0.0043,  0.1828,  0.1791,  0.0864,  0.2004,  0.1615,  0.1546,  0.0849,
         0.2048,  0.1482,  0.1423,  0.2155,  0.1491,  0.0229,  0.1282,  0.1698,
         0.1788,  0.1605,  0.1684,  0.1074,  0.1325,  0.1443,  0.1538,  0.2006,
         0.1235,  0.0872,  0.1562,  0.1862,  0.1447,  0.1506,  0.1580,  0.1830,
         0.1769,  0.2145,  0.1782,  0.1349,  0.1629,  0.1290,  0.1570,  0.1660,
         0.1405,  0.1155,  0.0628,  0.0567,  0.0105,  0.1334,  0.1739,  0.1087,
         0.0757,  0.0910,  0.0810,  0.07

## random traversal

In [41]:
# make term-to-path dict

def make_term_to_path_dict(term_to_path_dict, subtree, current_path):
    if subtree is None:
        return
    for term in subtree:
        term_path = [*current_path, term]
        #print(term, term_path, subtree[term])
        if term in term_to_path_dict:
            term_to_path_dict[term].append(term_path)
        else:
            term_to_path_dict[term] = [term_path]
        make_term_to_path_dict(term_to_path_dict, subtree[term], term_path)


term_to_path_dict = {}
make_term_to_path_dict(term_to_path_dict, IEEE_TAX, [])

In [42]:
def calc_f1(r,p):
    if r==0 and p==0:
        return 0
    return 2*p*r/(r+p)

def calc_random_prob_step(step, subtax, paths, stop_prob_factor):
    recall = 0
    precision = 0
    f1_score = 0

    p_stop = stop_prob_factor/(len(subtax)+stop_prob_factor)
    p_continue = len(subtax)/(len(subtax)+stop_prob_factor)
    p_specific_key = 1/(len(subtax)+stop_prob_factor)

    # if it stops:
    r = (step/(step+min([len(path) for path in paths])))
    p = 1
    f1 =  calc_f1(r,p)
    recall += p_stop * r
    precision += p_stop * p
    f1_score += p_stop * f1

    # if it continues
    for key in subtax.keys():
        key_paths = []
        for path in paths:

            # correct path:
            if path[0] == key:
                if len(path) == 1:
                    # path fully predicted
                    r=1
                    subtax_length = 0 if subtax[key] is None else len(subtax[key])
                    p_key_stop = stop_prob_factor/(subtax_length+stop_prob_factor)
                    prec_key_stop = 1
                    p_key_continue = (1-p_key_stop)
                    prec_key_continue = (step+1)/(step+2)
                    p = prec_key_stop*p_key_stop + prec_key_continue*p_key_continue
                    f1 = p_key_stop*calc_f1(r, prec_key_stop) + prec_key_continue*calc_f1(r, prec_key_continue)

                    f1_score += p_specific_key * f1
                    recall += p_specific_key * r
                    precision += p_specific_key * p
                else:
                    #print(path)
                    key_paths.append(path[1:])
            
        # if this key leads to any gt path
        if len(key_paths):
            r, p, f1 = calc_random_prob_step(step+1, subtax[key], key_paths, stop_prob_factor)
            recall += r*p_specific_key
            precision += p*p_specific_key
            f1_score += f1*p_specific_key
        
        # otherwise:
        else:
            r = step/(step+min([len(path) for path in paths]))
            p = step/(step+1)
            f1 = calc_f1(r, p)
            recall += p_specific_key * r
            precision += p_specific_key * p
            f1_score += p_specific_key * f1
    return recall, precision, f1_score
        

In [46]:
ignore_first = True#False


def calc_random_prob(paths, stop_prob_factor=1.0):
    if ignore_first:
        reduced_paths = []
        for path in paths:
            if len(path)>1:
                reduced_paths.append(path[1:])
        #print("red:", reduced_paths)
        recall, precision, f1_score = calc_random_prob_step(0, IEEE_TAX["Computational and artificial intelligence"], reduced_paths, stop_prob_factor)
    else:
        recall, precision, f1_score = calc_random_prob_step(0, IEEE_TAX, paths, stop_prob_factor)
    return recall, precision, f1_score


In [47]:
def get_results(stop_prob_factor=1.):
    results = pd.DataFrame(columns=["recall", "precision", "f1_score"])
    for key in df.index:
        gt_terms = df.loc[key, "IEEE Terms"]
        gt_terms = gt_terms.split(";")

        relevant_gt_terms_paths = []
        for gt_term in gt_terms:
            if gt_term in ieee_strings:    
                relevant_gt_terms_paths.extend(term_to_path_dict[gt_term])


        recall, precision, f1_score = calc_random_prob(relevant_gt_terms_paths, stop_prob_factor=stop_prob_factor)

        results.loc[key] = (recall, precision, f1_score)
    return results


In [48]:
for i in range(1,10):
    print(get_results(i/10).mean())

recall       0.129906
precision    0.147632
f1_score     0.119840
dtype: float64
recall       0.127389
precision    0.162154
f1_score     0.118173
dtype: float64
recall       0.124991
precision    0.176194
f1_score     0.116550
dtype: float64
recall       0.122700
precision    0.189776
f1_score     0.114968
dtype: float64
recall       0.120505
precision    0.202922
f1_score     0.113429
dtype: float64
recall       0.118398
precision    0.215653
f1_score     0.111929
dtype: float64
recall       0.116373
precision    0.227989
f1_score     0.110469
dtype: float64
recall       0.114423
precision    0.239948
f1_score     0.109047
dtype: float64
recall       0.112544
precision    0.251547
f1_score     0.107662
dtype: float64


# full ieee

In [69]:

def calc_random_prob_full(paths, stop_prob_factor=1.0):
    recall, precision, f1_score = calc_random_prob_step(0, IEEEFULL_TAX, paths, stop_prob_factor)
    return recall, precision, f1_score


In [75]:
def get_full_results(stop_prob_factor=1.):
    results = pd.DataFrame(columns=["recall", "precision", "f1_score"])
    for key in full_df.index:
        gt_terms = full_df.loc[key, "IEEE Terms"]
        gt_terms = gt_terms.split(";")
        
        relevant_gt_terms_paths = []
        for gt_term in gt_terms:
            if gt_term in ieeefull_strings:    
                relevant_gt_terms_paths.extend(full_term_to_path_dict[gt_term])


        recall, precision, f1_score = calc_random_prob(relevant_gt_terms_paths, stop_prob_factor=stop_prob_factor)

        results.loc[key] = (recall, precision, f1_score)
    return results


In [76]:
for i in range(1,10):
    print(get_full_results(i/10).mean())

recall       0.129906
precision    0.147632
f1_score     0.119840
dtype: float64
recall       0.127389
precision    0.162154
f1_score     0.118173
dtype: float64
recall       0.124991
precision    0.176194
f1_score     0.116550
dtype: float64
recall       0.122700
precision    0.189776
f1_score     0.114968
dtype: float64
recall       0.120505
precision    0.202922
f1_score     0.113429
dtype: float64
recall       0.118398
precision    0.215653
f1_score     0.111929
dtype: float64
recall       0.116373
precision    0.227989
f1_score     0.110469
dtype: float64
recall       0.114423
precision    0.239948
f1_score     0.109047
dtype: float64
recall       0.112544
precision    0.251547
f1_score     0.107662
dtype: float64


# lca_metric

In [49]:
from typing import Dict, List, Tuple, Optional

Path = List[str]
LabelToPaths = Dict[str, List[Path]]


def lca_depth_between_paths(path1: Path, path2: Path) -> Tuple[int, Optional[str]]:
    """
    Given two paths (root → … → leaf), return:
    - the depth (index) of their lowest common ancestor
    - the label of that LCA

    If they have no common prefix at all, returns (-1, None).
    """
    lca_depth = -1
    lca_label = None

    for i, (a, b) in enumerate(zip(path1, path2)):
        if a == b:
            lca_depth = i
            lca_label = a
        else:
            break

    return lca_depth, lca_label


def lca_depth_for_labels(
    label1: str, label2: str, label_to_paths: LabelToPaths
) -> Tuple[int, Optional[str]]:
    """
    Compute the deepest LCA over *all* path combinations between two labels in a DAG.
    Returns:
    - best_lca_depth (int, index in paths)
    - best_lca_label (str or None)
    """
    paths1 = label_to_paths.get(label1)
    paths2 = label_to_paths.get(label2)

    if not paths1:
        raise ValueError(f"No paths found for label {label1!r}")
    if not paths2:
        raise ValueError(f"No paths found for label {label2!r}")

    best_depth = -1
    best_label = None

    for p1 in paths1:
        for p2 in paths2:
            d, l = lca_depth_between_paths(p1, p2)
            if d > best_depth:
                best_depth = d
                best_label = l

    return best_depth, best_label


In [59]:
def label_depth_common_root(label: str, label_to_paths: LabelToPaths) -> int:
    """
    Depth in common-root setting.
    Assumes all paths for this label start with the same explicit root.
    Depth = index in path (root = 0).
    """
    paths = label_to_paths.get(label)
    if not paths:
        raise ValueError(f"No paths found for label {label!r}")
    # Longest path depth
    return max(len(p) - 1 for p in paths)


def lca_similarity_common_root(
    pred_label: str, true_label: str, label_to_paths: LabelToPaths
) -> float:
    """
    LCA-based similarity for taxonomies with a *named* common root in all paths.

    sim = depth(LCA) / max(depth(pred), depth(true))
    - exact match ⇒ 1.0
    - same parent/grandparent ⇒ in (0, 1)
    - only share the root ⇒ 0.0
    - different trees (shouldn't happen with a common root) ⇒ 0.0
    """
    d_pred = label_depth_common_root(pred_label, label_to_paths)
    d_true = label_depth_common_root(true_label, label_to_paths)
    max_depth = max(d_pred, d_true)

    if max_depth <= 0:
        return 0.0

    lca_d, _ = lca_depth_for_labels(pred_label, true_label, label_to_paths)

    if lca_d < 0:
        # No common prefix at all (weird if you truly have a single root, but safe)
        return 0.0

    return lca_d / max_depth


In [51]:
def label_depth_multi_root(
    label: str, label_to_paths: LabelToPaths, top_level_depth: int = 0
) -> int:
    """
    Depth in a multi-root / implicit-root setting.

    Paths start at the first real category (no shared root label).
    - If top_level_depth = 0 (default): top-level labels depth = 0, children = 1, etc.
    - If top_level_depth = 1: top-level labels depth = 1, children = 2, etc. (purely scaling)
    """
    paths = label_to_paths.get(label)
    if not paths:
        raise ValueError(f"No paths found for label {label!r}")

    # Depth index of last element in path
    depths = [len(p) - 1 + top_level_depth for p in paths]
    return max(depths)


def lca_similarity_multi_root(
    pred_label: str,
    true_label: str,
    label_to_paths: LabelToPaths,
    top_level_depth: int = 0,
) -> float:
    """
    LCA-based similarity for taxonomies with *no explicit shared root*.

    - If two labels live under the same top category, they will share
      some prefix and get a >0 similarity.
    - If they belong to different trees (e.g., 'Animals/...' vs 'Vehicles/...'),
      they will have no common prefix ⇒ similarity = 0.0.
    """
    d_pred = label_depth_multi_root(pred_label, label_to_paths, top_level_depth)
    d_true = label_depth_multi_root(true_label, label_to_paths, top_level_depth)
    max_depth = max(d_pred, d_true)

    if max_depth <= 0:
        return 0.0

    lca_d, _ = lca_depth_for_labels(pred_label, true_label, label_to_paths)

    if lca_d < 0:
        # Different trees: no common prefix at all
        return 0.0

    # lca_d is an index in the path, so we apply the same offset
    lca_depth = lca_d + top_level_depth

    return lca_depth / max_depth


In [52]:
def point_score_single_pred_common_root(
    pred_label: str,
    true_labels: List[str],
    label_to_paths: LabelToPaths,
) -> float:
    return max(
        lca_similarity_common_root(pred_label, t, label_to_paths)
        for t in true_labels
    )


def point_score_single_pred_multi_root(
    pred_label: str,
    true_labels: List[str],
    label_to_paths: LabelToPaths,
    top_level_depth: int = 0,
) -> float:
    return max(
        lca_similarity_multi_root(pred_label, t, label_to_paths, top_level_depth)
        for t in true_labels
    )


In [62]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieee_paths:
    pred = ieee_paths[key]
    gt_terms = df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieee_strings:
            rel_gt_terms.append(term)

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_common_root(pred, rel_gt_terms, term_to_path_dict)
    lca_results.loc[key] = score


In [63]:
lca_results

,lca
ieee_40938,0.666667
ieee_16802,1.000000
ieee_4042,0.333333
ieee_43881,0.333333
ieee_1785,1.000000
...,...
ieee_27010,0.000000
ieee_3580,0.000000
ieee_15688,1.000000
ieee_52407,0.500000


In [65]:
lca_results.mean()

lca    0.412811
dtype: float64

In [101]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieeefull_paths:
    pred = ieeefull_paths[key]
    gt_terms = full_df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieeefull_strings:
            rel_gt_terms.append(term)
    if len(rel_gt_terms)==0:
        print("none:", gt_terms)
        continue

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_common_root(pred, rel_gt_terms, full_term_to_path_dict)
    lca_results.loc[key] = score


none: ['Electrical engineering', 'Generators', 'Task analysis', 'Matlab']
none: ['Matlab', 'Mathematical model', 'Analytical models', 'Computational modeling', 'Engines', 'Data models', 'Integrated circuit modeling']


In [102]:
lca_results

,lca
fullieee_36325,0.000000
fullieee_22732,0.333333
fullieee_20107,0.000000
fullieee_16287,0.000000
fullieee_31647,0.000000
...,...
fullieee_43178,0.000000
fullieee_43391,0.000000
fullieee_12919,0.000000
fullieee_32103,0.000000


In [103]:
lca_results.mean()

lca    0.127437
dtype: float64

# absolute distance based metric

In [21]:
from typing import Dict, List, Tuple, Optional

Path = List[str]
LabelToPaths = Dict[str, List[Path]]


#def lca_depth_between_paths(path1: Path, path2: Path) -> Tuple[int, Optional[str]]:
#    """
#    Given two paths (root → … → leaf), return:
#    - the depth (index) of their lowest common ancestor
#    - the label of that LCA

#    If they have no common prefix at all, returns (-1, None).
#    """
#    lca_depth = -1
#    lca_label = None

#    for i, (a, b) in enumerate(zip(path1, path2)):
#        if a == b:
#            lca_depth = i
#            lca_label = a
#        else:
#            break
#
#    return lca_depth, lca_label

def lca_depth_between_paths(path1: Path, path2: Path) -> Tuple[int, Optional[str]]:
    """
    Given two paths (root → … → leaf), return:
    - the depth (index) of their lowest common ancestor
    - the label of that LCA
    """
    lca_depth = 0
    lca_label = None

    for i, (a, b) in enumerate(zip(path1, path2)):
        if a == b:
            lca_depth = i+1
            lca_label = a
        else:
            break

    return lca_depth, lca_label


def lca_depth_for_labels(
    label1: str, label2: str, label_to_paths: LabelToPaths
) -> Tuple[int, Optional[str]]:
    """
    Compute the deepest LCA over *all* path combinations between two labels in a DAG.
    Returns:
    - best_lca_depth (int, index in paths)
    - best_lca_label (str or None)
    """
    paths1 = label_to_paths.get(label1)
    paths2 = label_to_paths.get(label2)

    if not paths1:
        raise ValueError(f"No paths found for label {label1!r}")
    if not paths2:
        raise ValueError(f"No paths found for label {label2!r}")

    best_depth = -1
    best_label = None

    for p1 in paths1:
        for p2 in paths2:
            d, l = lca_depth_between_paths(p1, p2)
            if d > best_depth:
                best_depth = d
                best_label = l

    return best_depth, best_label


def shortest_ontology_distance(
    label1: str, label2: str, label_to_paths: LabelToPaths
) -> Optional[int]:
    """
    Shortest path length (number of edges) between two labels in the ontology DAG,
    based on their root→label paths.

    Returns:
        - d (int) = minimum distance over all path combinations, if any common ancestor
        - None if the labels live in completely separate trees (no common prefix).
    """
    paths1 = label_to_paths.get(label1)
    paths2 = label_to_paths.get(label2)

    if not paths1:
        raise ValueError(f"No paths found for label {label1!r}")
    if not paths2:
        raise ValueError(f"No paths found for label {label2!r}")

    best_dist: Optional[int] = None

    for p1 in paths1:
        dp = len(p1)
        for p2 in paths2:
            dt = len(p2)
            lca_d, _ = lca_depth_between_paths(p1, p2)
            if lca_d < 0:
                # no common prefix for this path pair
                continue

            # distance via this LCA
            d = (dp - lca_d) + (dt - lca_d)

            if best_dist is None or d < best_dist:
                best_dist = d

    return best_dist


def distance_similarity(
    pred_label: str, true_label: str, label_to_paths: LabelToPaths
) -> float:
    """
    Distance-based similarity:

        sim(p, t) = 1 / (1 + d)

    where d is the shortest ontology distance between pred_label and true_label.

    - exact match (same node, same path) → d = 0 → sim = 1.0
    - one edge apart → d = 1 → sim = 0.5
    - further apart → smaller sim
    - no common ancestor (different trees) → sim = 0.0
    """
    d = shortest_ontology_distance(pred_label, true_label, label_to_paths)
    #if d is None:
    #    # disconnected components / different trees
    #    return 0.0
    return 1.0 / (1.0 + d)


from typing import Iterable


def point_score_single_pred_distance(
    pred_label: str,
    true_labels: Iterable[str],
    label_to_paths: LabelToPaths,
) -> float:
    """
    Score for one data point with:
    - one predicted label
    - several valid ground-truth labels

    We take the best similarity to any valid label.
    """
    return max(
        distance_similarity(pred_label, t, label_to_paths)
        for t in true_labels
    )


In [34]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieee_paths:
    pred = ieee_paths[key]
    gt_terms = df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieee_strings:
            rel_gt_terms.append(term)

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_distance(pred, rel_gt_terms, term_to_path_dict)
    lca_results.loc[key] = score


In [35]:
lca_results.mean()

lca    0.459185
dtype: float64

In [36]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieeefull_paths:
    pred = ieeefull_paths[key]
    gt_terms = full_df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieeefull_strings:
            rel_gt_terms.append(term)
    if len(rel_gt_terms)==0:
        print("none:", gt_terms)
        continue

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_distance(pred, rel_gt_terms, full_term_to_path_dict)
    lca_results.loc[key] = score


none: ['Electrical engineering', 'Generators', 'Task analysis', 'Matlab']
none: ['Matlab', 'Mathematical model', 'Analytical models', 'Computational modeling', 'Engines', 'Data models', 'Integrated circuit modeling']


In [37]:
lca_results.mean()

lca    0.25969
dtype: float64

# others

In [18]:
def distance_similarity_linear(
    pred_label: str,
    true_label: str,
    label_to_paths,
    alpha: float = 0.2,
) -> float:
    """
    Linear distance-based similarity:

        sim = max(0, 1 - alpha * d)

    where d is the shortest ontology distance between pred_label and true_label.

    - exact match (d = 0) → sim = 1.0
    - 1 edge apart → sim = 1 - alpha
    - 2 edges apart → sim = 1 - 2*alpha
    - etc.
    - large distances eventually clamp at 0.
    """
    d = shortest_ontology_distance(pred_label, true_label, label_to_paths)
    if d is None:
        return 0.0

    s = 1.0 - alpha * d
    return s if s > 0.0 else 0.0

def distance_similarity_stretched_inverse(
    pred_label: str,
    true_label: str,
    label_to_paths,
    k: float = 3.0,
) -> float:
    """
    Stretched inverse distance:

        sim = 1 / (1 + d / k)

    with k > 1 giving a softer drop-off than 1 / (1 + d).

    Example with k=3:
        d=0 → 1.0
        d=1 → ~0.75
        d=2 → ~0.60
        d=3 → 0.5
    """
    d = shortest_ontology_distance(pred_label, true_label, label_to_paths)
    if d is None:
        return 0.0

    return 1.0 / (1.0 + d / k)

def point_score_single_pred_linear(
    pred_label: str,
    true_labels,
    label_to_paths,
    alpha: float = 0.2,
) -> float:
    return max(
        distance_similarity_linear(pred_label, t, label_to_paths, alpha)
        for t in true_labels
    )


def point_score_single_pred_stretched(
    pred_label: str,
    true_labels,
    label_to_paths,
    k: float = 3.0,
) -> float:
    return max(
        distance_similarity_stretched_inverse(pred_label, t, label_to_paths, k)
        for t in true_labels
    )


In [39]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieee_paths:
    pred = ieee_paths[key]
    gt_terms = df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieee_strings:
            rel_gt_terms.append(term)

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]




    score = point_score_single_pred_linear(pred, rel_gt_terms, term_to_path_dict, 1/6)
    lca_results.loc[key, "lin"] = score
    score = point_score_single_pred_stretched(pred, rel_gt_terms, term_to_path_dict, 2.5)
    lca_results.loc[key, "str"] = score
print(lca_results.mean())

lca         NaN
lin    0.586714
str    0.601892
dtype: object


In [40]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieeefull_paths:
    pred = ieeefull_paths[key]
    gt_terms = full_df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieeefull_strings:
            rel_gt_terms.append(term)
    if len(rel_gt_terms)==0:
        print("none:", gt_terms)
        continue

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_linear(pred, rel_gt_terms, full_term_to_path_dict, 1/8)
    lca_results.loc[key, "lin"] = score
    score = point_score_single_pred_stretched(pred, rel_gt_terms, full_term_to_path_dict, 3.0)
    lca_results.loc[key, "str"] = score
print(lca_results.mean())

none: ['Electrical engineering', 'Generators', 'Task analysis', 'Matlab']
none: ['Matlab', 'Mathematical model', 'Analytical models', 'Computational modeling', 'Engines', 'Data models', 'Integrated circuit modeling']
lca         NaN
lin    0.477961
str    0.463153
dtype: object


# less than k  gives 1

In [19]:
def distance_thr(
    pred_label: str,
    true_label: str,
    label_to_paths,
    k: int = 1,
) -> float:
    """
    Stretched inverse distance:

        sim = 1 / (1 + d / k)

    with k > 1 giving a softer drop-off than 1 / (1 + d).

    Example with k=3:
        d=0 → 1.0
        d=1 → ~0.75
        d=2 → ~0.60
        d=3 → 0.5
    """
    d = shortest_ontology_distance(pred_label, true_label, label_to_paths)
    if d<= k:
        return 1
    return 0

def point_score_single_pred_thr(
    pred_label: str,
    true_labels,
    label_to_paths,
    k: float = 3.0,
) -> float:
    return max(
        distance_thr(pred_label, t, label_to_paths, k)
        for t in true_labels
    )

In [22]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieee_paths:
    pred = ieee_paths[key]
    gt_terms = df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieee_strings:
            rel_gt_terms.append(term)

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]



    score = point_score_single_pred_thr(pred, rel_gt_terms, term_to_path_dict, 0)
    lca_results.loc[key, "0"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, term_to_path_dict, 1)
    lca_results.loc[key, "1"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, term_to_path_dict, 2)
    lca_results.loc[key, "2"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, term_to_path_dict, 3)
    lca_results.loc[key, "3"] = score
for key in ieeefull_paths:
    pred = ieeefull_paths[key]
    gt_terms = full_df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieeefull_strings:
            rel_gt_terms.append(term)
    if len(rel_gt_terms)==0:
        print("none:", gt_terms)
        continue

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 0)
    lca_results.loc[key, "f0"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 1)
    lca_results.loc[key, "f1"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 2)
    lca_results.loc[key, "f2"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 3)
    lca_results.loc[key, "f3"] = score
lca_results.mean()

none: ['Electrical engineering', 'Generators', 'Task analysis', 'Matlab']
none: ['Matlab', 'Mathematical model', 'Analytical models', 'Computational modeling', 'Engines', 'Data models', 'Integrated circuit modeling']


lca         NaN
0       0.26121
1      0.378648
2      0.498932
3      0.640569
f0     0.067294
f1       0.1196
f2     0.169556
f3      0.30767
dtype: object

# correct at level

In [45]:
import numpy as np
def check_level(p,t,l):
    if len(t)<l+1:
        return 0.5
    if len(p)<l+1:
        return 0
    return int(p[l] == t[l])

def correct_at_level(
    pred_label: str,
    true_label: str,
    label_to_paths,
    level: int = 1,
) -> float:

    predpaths = label_to_paths[pred_label]
    truepaths = label_to_paths[true_label]

    score = max([max([check_level(p1,p2,level) for p1 in predpaths]) for p2 in truepaths])
    if score == 0.5:
        return np.nan
    else:
        return score



def level_single(
    pred_label: str,
    true_labels,
    label_to_paths,
    level: float = 3.0,
) -> float:
    return max(
        correct_at_level(pred_label, t, label_to_paths, level)
        for t in true_labels
    )

In [46]:
lca_results = pd.DataFrame(columns=["lca"])
for key in ieee_paths:
    pred = ieee_paths[key]
    gt_terms = df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieee_strings:
            rel_gt_terms.append(term)

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]



    score = level_single(pred, rel_gt_terms, term_to_path_dict, 0)
    lca_results.loc[key, "0"] = score
    score = level_single(pred, rel_gt_terms, term_to_path_dict, 1)
    lca_results.loc[key, "1"] = score
    score = level_single(pred, rel_gt_terms, term_to_path_dict, 2)
    lca_results.loc[key, "2"] = score
    score = level_single(pred, rel_gt_terms, term_to_path_dict, 3)
    lca_results.loc[key, "3"] = score
for key in ieeefull_paths:
    pred = ieeefull_paths[key]
    gt_terms = full_df.loc[key, "IEEE Terms"]

    gt_terms = gt_terms.split(";")
    rel_gt_terms = []
    for term in gt_terms:
        if term in ieeefull_strings:
            rel_gt_terms.append(term)
    if len(rel_gt_terms)==0:
        print("none:", gt_terms)
        continue

    pred = pred[-2] if pred[-1] == "Other" else pred[-1]

    #print(pred)
    #print(gt_terms)

    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 0)
    lca_results.loc[key, "f0"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 1)
    lca_results.loc[key, "f1"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 2)
    lca_results.loc[key, "f2"] = score
    score = point_score_single_pred_thr(pred, rel_gt_terms, full_term_to_path_dict, 3)
    lca_results.loc[key, "f3"] = score
lca_results.mean()

none: ['Electrical engineering', 'Generators', 'Task analysis', 'Matlab']
none: ['Matlab', 'Mathematical model', 'Analytical models', 'Computational modeling', 'Engines', 'Data models', 'Integrated circuit modeling']


lca         NaN
0           1.0
1      0.585053
2      0.356542
3      0.213174
f0     0.067294
f1       0.1196
f2     0.169556
f3      0.30767
dtype: object

In [40]:
lca_results.mean() # nan is 1 (upper limit)

lca         NaN
0           1.0
1      0.645552
2      0.580071
3      0.884698
f0     0.067294
f1       0.1196
f2     0.169556
f3      0.30767
dtype: object

In [ ]:
lca_results.mean() # nan is 0

lca         NaN
0           1.0
1      0.585053
2      0.347331
3      0.145196
f0     0.067294
f1       0.1196
f2     0.169556
f3      0.30767
dtype: object

In [ ]:
max([0,np.nan])

0